# Demo of a Large Language Model Application

This application demonstrates an LLM based customer churn assistant that uses OpenAI Agents SDK and MCP tools to analyze telecom customer churn risk. The assistant connects a large language model with external tools that retrieve customer data, predict churn probability and return retention offers. The goal is to show how an LLM can act as an intelligent interface between technical systems and business users.

## Import required libraries

This section imports the Python libraries needed for the demo.  
- `os` is used to read environment variables such as the API key.  
- `sys` is used when starting the local MCP server process.  
- `Agent`, `Runner`, and `ModelSettings` come from the OpenAI Agents SDK and are used to define and run the agents.  
- `MCPServerStdio` is used to connect the LLM agent to the local MCP server over standard I/O.

In [43]:
import os
import sys

from agents import Agent, Runner, ModelSettings, trace
from agents.mcp import MCPServerStdio

## Check environment setup

This section checks whether the `OPENAI_API_KEY` environment variable is available. The notebook requires a valid OpenAI API key in order to send requests to the language model. This is a simple validation step before running the rest of the workflow.

In [45]:
if not os.getenv("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY is not set. Set it before running the agent cells.")
else:
    print("OPENAI_API_KEY is set.")

OPENAI_API_KEY is set.


## Define the agents

This section creates two agents for the application.

- The **Retention Strategist** is the main specialist agent. Its role is to analyze customer churn risk, identify churn drivers and recommend retention offers.
- The **Triage Agent** acts as the front door. It receives the user request and decides to answer directly or hand the task off to the specialist agent.

This design shows a simple multi-agent pattern where one agent manages routing and another performs the domain specific task.

In [46]:
retention_strategist = Agent(
    name="Retention Strategist",
    model="gpt-5",
    instructions="""
You are a telecom retention strategist.

You must use tools before making claims.
For any customer analysis:
1. Get the customer profile.
2. Predict churn.
3. Get retention offers for the returned risk segment.

Then produce:
- Risk level
- Main churn drivers
- Recommended offer(s)
""".strip(),
    model_settings=ModelSettings(tool_choice="required"),
)

triage_agent = Agent(
    name="Triage Agent",
    model="gpt-5",
    instructions="""
You are the front-door agent for a telecom customer churn analysis assistant.

Your job is to handle requests related ONLY to:
1. Telecom customer churn analysis
2. The Telco Customer Churn dataset used in this demo
3. Customer churn predictions generated by the churn model
4. Retention offers or churn mitigation strategies
5. The tools and agents available in this application

Allowed requests include:
- Analyzing a specific telecom customer’s churn risk
- Summarizing the Telco churn dataset
- Listing valid customer IDs
- Explaining churn predictions
- Recommending retention actions
- Describing the available MCP tools or agents

If a request is related to customer churn analysis, hand the task off to the **Retention Strategist** agent.

If the request asks about:
- the dataset
- available tools
- customer IDs
you may answer directly using the available tools.

If the request is unrelated to telecom churn, customer data, or the system’s tools and agents,
politely refuse and explain that this assistant only supports telecom customer churn analysis.

Example refusal:
"I'm designed to assist with telecom customer churn analysis and questions related to this demo system. I cannot answer unrelated questions."

Always use the MCP tools when factual information about customers or the dataset is required.
""".strip(),
    handoffs=[retention_strategist]
)

## Churn assistant

This section defines the main function that runs the churn assistant. The function starts a connection to the local MCP server and makes the MCP tools available to the agents.

The MCP server acts as the bridge between the LLM and external capabilities like:
- Retrieving customer profiles.
- Predicting churn risk using the trained model.
- Fetching available retention offers.

The function then runs the triage agent, which may hand off the request to the retention strategist when needed.

In [47]:
async def churn_assistant(prompt: str) -> str:
    async with MCPServerStdio(
        name="Telco MCP Server",
        params={
            "command": sys.executable,
            "args": [str("telco_churn_server.py")],
            "env": dict(os.environ),
        },
    ) as server:
        strategist_with_tools = retention_strategist.clone(mcp_servers=[server])
        triage_with_tools = triage_agent.clone(
            mcp_servers=[server],
            handoffs=[strategist_with_tools]
        )
        with trace("telco_churn_assistant"):
            result = await Runner.run(triage_with_tools, prompt)
            return result.final_output

## Test the tool connection

This step asks the assistant to list all available tools. It is mainly a validation step to confirm that the agent is successfully connected to the MCP server and can see the tools exposed by that server.

In [48]:
response = await churn_assistant("List all available tools.")
print(response)

Here are the available tools and agents in this application:

Tools (functions namespace)
- dataset_overview: Returns dataset shape, target balance, and sample columns.
- list_customer_ids(limit?): Returns a sample list of customer IDs for testing.
- get_customer_profile(customer_id): Returns the raw CSV row for a specific customer.
- predict_churn(customer_id): Predicts churn probability using the trained logistic regression model.
- get_retention_offers(risk_segment): Returns retention offers for Low, Medium, or High risk segments.
- transfer_to_retention_strategist: Hands off the request to the Retention Strategist agent.

Tool wrapper
- multi_tool_use.parallel: Runs multiple functions tools in parallel when they can operate concurrently.

Agent
- Retention Strategist: Specialized agent for handling churn-risk analysis and recommending retention actions.


## Summarize the dataset

This section shows how the churn assistant summarizes the dataset that powers the demo. Instead of hardcoding dataset details, the LLM uses MCP tools to retrieve real information such as the number of rows, columns, churn rate and example features.

The assistant then converts this structured data into a natural language summary that is easy to understand. This highlights one of the key strengths of LLM based systems: transforming raw data into human readable insights.

This example also shows how the LLM acts as an interface between the user and the dataset. The user asks a simple natural language question and the assistant coordinates tool calls in the background to generate response.


In [49]:
response = await churn_assistant("Summarize the dataset that powers this churn demo.")
print(response)

Here’s a concise overview of the Telco Customer Churn dataset used in this demo:

- Size: 7,032 rows and 21 columns
- Target balance: Churn rate is 26.58% (roughly 27% churners, 73% non-churners)
- Example columns: customerID, gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines, InternetService, OnlineSecurity
- Model: A Logistic Regression model powers the churn predictions in this demo

Columns capture customer demographics and subscribed services (e.g., phone and internet options) along with tenure-related attributes. If you want, I can list all columns or break down feature groups.


## Retrieve example customer IDs

This section asks the assistant to list valid customer IDs from the dataset.

In [50]:
response = await churn_assistant("List 5 valid customer IDs from the dataset.")
print(response)

Here are 5 valid customer IDs:
- 7590-VHVEG
- 5575-GNVDE
- 3668-QPYBK
- 7795-CFOCW
- 9237-HQITU


## Retrieve available retention offers

This section asks the assistant to list all retention offers available through the MCP tools. These offers are later used by the LLM when generating recommendations for medium risk or high risk customers.

In [51]:
response = await churn_assistant("List all available retention offers.")
print(response)

Here are the available retention offers by risk segment:

- Low-risk
  - Standard loyalty communication
  - Periodic engagement reminder
  - No aggressive incentive required

- Medium-risk
  - 10% discount for 3 months
  - Bundle upgrade offer
  - Loyalty points campaign

- High-risk
  - 20% discount for 6 months
  - Free tech support for 3 months
  - Annual contract upgrade incentive

If you’d like, I can also recommend which set to use for a specific customer—just share their customer ID.


## Customer churn analysis examples

This section demonstrates a full end-to-end churn analysis for a specific customer using the LLM powered assistant. The assistant receives a prompt, retrieves the customer profile and churn prediction through MCP tools and generates a user friendly explanation.

The output includes:
- the predicted churn risk level based on the model probability.
- key factors contributing to the risk.
- And recommended retention actions.

This example highlights how the LLM combines model outputs and customer attributes to produce meaningful response. Instead of returning raw data or probabilities, the assistant translates them into recommendations that business users can easily understand and apply.



#### Define the customer analysis prompt

This section creates a reusable prompt template for customer analysis. The template asks the assistant to:
- Analyze a specific customer.
- Determine the churn risk level.
- Identify main churn drivers when appropriate.
- Recommend retention offers.

In [52]:
prompt_template = """
Analyze customer {0} using the real dataset and prediction tool.
Provide a risk level, main churn drivers when churn level is high or medium, and recommended retention offers.
""".strip()

#### Example 1: Analyze customer 7590-VHVEG

In [53]:
customer_id = "7590-VHVEG"
prompt = prompt_template.format(customer_id)
response = await churn_assistant(prompt)
print(response)

- Risk level: Medium (predicted churn probability: 0.64)

- Main churn drivers:
  - Month-to-month contract (highest churn-prone contract type)
  - Very short tenure (1 month) and very low total charges (early lifecycle churn risk)
  - Electronic check payment and paperless billing (both correlate with higher churn in this dataset)
  - Limited value-add/support features enabled (no Online Security, no Tech Support), which can reduce perceived value and stickiness

- Recommended retention offers:
  - 10% discount for 3 months — use as an immediate cost relief to stabilize satisfaction during the onboarding window
  - Bundle upgrade offer — position an Internet + value-add bundle (e.g., add Online Security and/or Tech Support to DSL) to increase perceived value and reduce hassle of switching
  - Loyalty points campaign — pair with a simple, near-term redemption to encourage engagement and extend commitment beyond the first few billing cycles


#### Example 2: Analyze customer 4929-XIHVW

In [12]:
customer_id = "4929-XIHVW"
prompt = prompt_template.format(customer_id)
response = await churn_assistant(prompt)
print(response)

Here’s the analysis for customer 4929-XIHVW based on the live dataset and prediction tool:

- Risk level: High (predicted churn probability: 0.8429)

- Main churn drivers:
  - Month-to-month contract combined with very short tenure (2 months), a strong early-life churn signal.
  - High monthly charges ($95.50) relative to tenure, indicating immediate price sensitivity.
  - Fiber optic internet without value-add support features: No OnlineSecurity, No TechSupport, No OnlineBackup.
  - Paperless billing is present, which in this model aligns with higher churn segments (secondary factor).

- Recommended offer(s):
  - 20% discount for 6 months to address near-term price concerns and stabilize satisfaction during early tenure.
  - Free tech support for 3 months to raise perceived value and reduce friction given the absence of TechSupport.
  - Annual contract upgrade incentive to transition off month-to-month; position after initial discount acceptance to improve stickiness.


#### Example 3: Analyze customer 8191-XWSZG

In [54]:
customer_id = "8191-XWSZG"
prompt = prompt_template.format(customer_id)
response = await churn_assistant(prompt)
print(response)


- Risk level: Low (predicted churn probability: 11.2%)

- Recommended offer(s):
  - Standard loyalty communication
  - Periodic engagement reminder
  - No aggressive incentive required


#### Example 4: Analyze customer 4445-ZJNMU

In [ ]:
customer_id = "4445-ZJNMU"
prompt = prompt_template.format(customer_id)
response = await churn_assistant(prompt)
print(response)

Here’s the analysis for customer 4445-ZJNMU based on the live dataset and prediction tool.

- Risk level: High (predicted churn probability: 79.2%)

- Main churn drivers:
  - Month-to-month contract, which is more churn-prone than term contracts.
  - Fiber optic internet without OnlineSecurity and TechSupport, a combination commonly linked to higher churn.
  - High monthly charges ($99.30) with several add-ons (e.g., StreamingTV/Movies), paired with relatively short tenure (9 months).

- Recommended retention offers:
  - 20% discount for 6 months
  - Free tech support for 3 months
  - Annual contract upgrade incentive

Recommended approach: Lead with the 20% discount plus 3 months of free tech support to immediately address price sensitivity and support gaps, then position the annual contract upgrade incentive once satisfaction improves.


#### Example 5: Analyze customer 6047-YHPVI

In [8]:
customer_id = "6047-YHPVI"
prompt = prompt_template.format(customer_id)
response = await churn_assistant(prompt)
print(response)

Here’s the analysis for customer 6047-YHPVI:

- Risk level: Medium (predicted churn probability: 0.727)

- Main churn drivers:
  - Month-to-month contract (no commitment increases churn risk)
  - Very short tenure (5 months)
  - Electronic check and paperless billing (payment method pattern often linked with higher churn)
  - Fiber optic internet with relatively high monthly charges ($69.70)
  - No value-added protection/support services (OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport all No)

- Recommended offer(s):
  - 10% discount for 3 months to ease near-term price sensitivity
  - Bundle upgrade offer (e.g., add OnlineSecurity and TechSupport to the current fiber plan at a promo rate to increase perceived value and stickiness)
  - Loyalty points campaign to boost engagement and reward continued tenure


## Compare retention recommendations for two customers

In this section, the assistant is asked to compare the retention strategies recommended for two different customers: **7590-VHVEG** and **4445-ZJNMU**.

The goal of this step is to show how the LLM can analyze multiple customer profiles and explain the differences in recommended actions based on their churn risk levels and attributes. The assistant uses the MCP tools to retrieve each customer’s profile, predict churn risk using the churn model and determine appropriate retention offers.

This example highlights the LLM’s ability to perform **comparative reasoning** and generate a natural language explanation describing why different retention strategies may be suggested for different customers.

In [55]:
prompt = "Explain how the retention offer recommendations differ for customers 7590-VHVEG and 4445-ZJNMU."
response = await churn_assistant(prompt)
print(response)

Here’s how the recommendations differ once we analyze each customer.

Customer 7590-VHVEG
- Risk level: Medium (predicted churn probability ~0.64)
- Main churn drivers:
  - Very short tenure (1 month) with a month-to-month contract
  - Limited product attachment (DSL only, no phone, no security/tech support)
  - Electronic check payment (often correlates with higher churn)
- Recommended offers:
  - 10% discount for 3 months
  - Bundle upgrade offer
  - Loyalty points campaign

Customer 4445-ZJNMU
- Risk level: High (predicted churn probability ~0.79)
- Main churn drivers:
  - Month-to-month contract with relatively high monthly charges (~$99)
  - Fiber optic with multiple add-ons (streaming), but lacking TechSupport/OnlineSecurity
  - Mid-short tenure (9 months), potential price/value sensitivity
- Recommended offers:
  - 20% discount for 6 months
  - Free tech support for 3 months
  - Annual contract upgrade incentive

How the recommendations differ
- Depth and duration of discount: H

## Explain how the churn model predicts risk

In this step, the assistant is asked to explain how the churn prediction model determines risk for a specific customer. The question focuses on the model’s reasoning process, instead of just returning a prediction. 

The LLM uses the available tools to retrieve the customer's profile and churn prediction, and then converts the output of the churn model into a clear explanation in natural language. This shows how a large language model can help business users understand model predictions without needing technical knowledge of the underlying algorithm.

In [24]:
prompt = "Explain how model is predicting churn risk for customer 7590-VHVEG."
response = await churn_assistant(prompt)
print(response)

Here’s how the model rates customer 7590-VHVEG and why, based on their actual profile and the model’s output.

Risk level
- Medium risk (predicted churn probability: 0.64; model: DecisionTreeClassifier)

Main churn drivers (from this customer’s profile)
- Very short tenure (1 month): New customers are more volatile and have not yet built loyalty.
- Contract: Month-to-month, which is strongly associated with higher churn versus term contracts.
- Payment/billing: Electronic check with paperless billing, a pattern commonly linked to higher churn in our historical data.
- Limited stickiness features: No OnlineSecurity and no TechSupport; fewer value-add services reduce switching costs.
- Mitigating signals: DSL (typically lower churn than fiber), low monthly charges, and having a partner slightly offset risk, but not enough to overcome the above factors.

Recommended offer(s) for this risk segment (Medium)
- 10% discount for 3 months
- Bundle upgrade offer (position security/support add-on

## Prompt variation experiment

In this section, the prompt wording is modified to observe how the LLM responds to slightly different instructions. Even though the underlying data and tools remain the same, changing the phrasing of the prompt can influence how the model structures its explanation.

The goal of this experiment is to evaluate if the assistant produces consistent reasoning and recommendations when the request is phrased differently. This helps test the quality of the prompt design and shows how prompt engineering can impact LLM output quality.

The assistant will analyze the same customers but will be asked to provide the explanation in a slightly different format.

In [56]:
prompt = """
Compare the churn risk and retention strategies for customers 7590-VHVEG and 4445-ZJNMU.

Explain:
1. Which customer has higher churn risk
2. Why their risk levels differ
3. How the recommended retention offers differ
"""

response = await churn_assistant(prompt)
print(response)

Here’s a side-by-side churn assessment based on profiles, model predictions, and risk-segment offers.

Customer 7590-VHVEG
- Risk level: Medium (predicted churn probability 0.64)
- Main churn drivers:
  - Very short tenure (1 month) and month-to-month contract
  - Electronic check payment and paperless billing (often associated with higher churn)
  - Few value-adding services (no phone, no security/tech support), low switching cost
- Recommended offers (Medium-risk set):
  - Bundle upgrade offer (add OnlineSecurity/TechSupport to increase stickiness)
  - 10% discount for 3 months to ease early churn risk
  - Loyalty points campaign to build engagement

Customer 4445-ZJNMU
- Risk level: High (predicted churn probability 0.79)
- Main churn drivers:
  - Month-to-month contract with relatively short tenure (9 months)
  - High monthly charges ($99.30) driven by fiber optic plus multiple streaming add-ons (price sensitivity)
  - Lacking OnlineSecurity and TechSupport (service-quality gap)
  

## Handling unrelated questions

This section demonstrates how the churn assistant behaves when a user asks a question that is outside the scope of the application. The assistant is intentionally designed to support only telecom customer churn analysis and questions related to the dataset, tools and agents used in this demo.

When the user asks an unrelated question (for example, asking about general AI tasks), the triage agent detects that the request is not relevant to telecom churn analysis. Instead of attempting to answer the question, the assistant declines the request and explains that it is limited to churn related functionality.

This shows how domain restrictions can be implemented in LLM applications to prevent the system from responding to unrelated prompts. It also helps keep the assistant focused on its designed purpose and ensures that responses are relevant to the telecom churn analysis use case.

In [58]:
prompt = "What were the main reasons for the fall of the Roman Empire?"
response = await churn_assistant(prompt)
print(response)

I’m designed to assist with telecom customer churn analysis and questions related to this demo system. I can’t answer unrelated questions like the fall of the Roman Empire.

If you’d like, I can:
- Summarize the Telco Customer Churn dataset
- List valid customer IDs
- Analyze a specific customer’s churn risk (and hand off to the Retention Strategist)
- Recommend retention offers or mitigation strategies
